# FABQ-VP: Variable Precision Quantization for 8B Models

**Goal:** Test FABQ-VP on an 8B model targeting ~3 bpw with <5% perplexity degradation

Based on research plan: Variable Precision Extension extending FABQ-RC's adaptive precision from 1-bit to 2-8 bits

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
from tqdm import tqdm
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load Model and Tokenizer

Use Qwen2.5-8B as target 8B model

In [ ]:
model_name = "Qwen/Qwen2.5-8B"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

num_params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded: {num_params:.2f}B parameters")

## 2. Calibration Dataset

Load WikiText2 for calibration

In [ ]:
from datasets import load_dataset

def get_calibration_data(tokenizer, num_samples=512):
    ds = load_dataset("wikitext", "wikitext-2-v1", split="train")
    texts = [ds[i]["text"] for i in range(min(num_samples, len(ds)))]
    enc = tokenizer("\n".join(texts), return_tensors="pt", truncation=True, max_length=2048)
    return enc.input_ids

calib_ids = get_calibration_data(tokenizer)
print(f"Calibration tokens: {calib_ids.shape[1]}")

## 3. FP16 Baseline Perplexity

In [ ]:
def compute_perplexity(model, input_ids, seq_len=128):
    losses = []
    max_start = max(0, input_ids.shape[1] - seq_len - 1)
    for i in range(0, max_start, seq_len):
        batch = input_ids[:, i:i+seq_len].to(device)
        labels = batch.clone()
        with torch.no_grad():
            outputs = model(batch, labels=labels)
            losses.append(outputs.loss.item())
    if not losses:
        return float('inf')
    return np.exp(np.mean(losses))

print("Computing FP16 baseline perplexity...")
fp16_ppl = compute_perplexity(model, calib_ids)
print(f"FP16 Perplexity: {fp16_ppl:.4f}")

## 4. FABQ-VP: Variable Precision Pyramid

Implement 5-level precision allocation:
- fp16: top 0.5% channels (layernorm, critical)
- int8: next 4.5% channels (sensitive)
- int4: next 20% channels (medium importance)
- int2: next 25% channels (low importance)
- binary: remaining 50% (least important)

In [ ]:
class FABQVPAllocator:
    def __init__(self, pyramid=None):
        self.pyramid = pyramid or {
            'fp16': 0.005,
            'int8': 0.045,
            'int4': 0.200,
            'int2': 0.250,
            'binary': 0.500
        }
        self.precisions = ['fp16', 'int8', 'int4', 'int2', 'binary']
        self.fisher_scores = {}
        self.allocations = {}
    
    def compute_fisher_importance(self, model, calib_ids):
        print("Computing Fisher importance...")
        model.eval()
        
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                module._fisher_acc = torch.zeros(module.out_features, device=device)
        
        num_tokens = calib_ids.shape[1]
        for i in tqdm(range(0, num_tokens - 128, 128)):
            batch = calib_ids[:, i:i+128].to(device)
            labels = batch.clone()
            
            model.zero_grad()
            outputs = model(batch, labels=labels)
            loss = outputs.loss
            loss.backward()
            
            for name, module in model.named_modules():
                if isinstance(module, nn.Linear):
                    if hasattr(module, 'weight') and module.weight.grad is not None:
                        grad_sq = module.weight.grad.data ** 2
                        module._fisher_acc += grad_sq.sum(dim=1)
            
            model.zero_grad()
        
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                self.fisher_scores[name] = module._fisher_acc.clone()
        
        print("Fisher computation complete")
        return self.fisher_scores
    
    def allocate_precision(self, name):
        if name not in self.fisher_scores:
            return {}
        scores = self.fisher_scores[name]
        n_channels = scores.shape[0]
        sorted_indices = torch.argsort(scores, descending=True)
        
        allocation = {}
        cumulative = 0
        
        for precision in self.precisions:
            fraction = self.pyramid[precision]
            threshold = int(n_channels * fraction)
            for i in range(threshold):
                if cumulative + i >= n_channels:
                    break
                idx = sorted_indices[cumulative + i].item()
                allocation[idx] = precision
            cumulative += threshold
            if cumulative >= n_channels:
                break
        
        return allocation
    
    def run_full_allocation(self, model, calib_ids):
        self.compute_fisher_importance(model, calib_ids)
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear) and name in self.fisher_scores:
                self.allocations[name] = self.allocate_precision(name)
        return self.allocations

allocator = FABQVPAllocator()

## 5. Compute Fisher Importance (subset for speed)

In [ ]:
fisher_allocator = FABQVPAllocator()
fisher_allocator.compute_fisher_importance(model, calib_ids[:, :2048])

allocations = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) and name in fisher_allocator.fisher_scores:
        allocations[name] = fisher_allocator.allocate_precision(name)

precision_counts = {'fp16': 0, 'int8': 0, 'int4': 0, 'int2': 0, 'binary': 0}
for name, alloc in allocations.items():
    for prec in alloc.values():
        precision_counts[prec] += 1

total = sum(precision_counts.values())
print("Precision allocation summary:")
for prec, count in precision_counts.items():
    print(f"  {prec}: {count} channels ({100*count/total:.1f}%)")

## 6. Quantization Functions

In [ ]:
def quantize_channel(ch_weights, precision, block_size=128):
    if precision == 'fp16':
        return ch_weights
    elif precision == 'int8':
        scale = ch_weights.abs().amax() / 127
        return torch.clamp(torch.round(ch_weights / scale), -128, 127) * scale
    elif precision == 'int4':
        scale = ch_weights.abs().amax() / 7.5
        return torch.clamp(torch.round(ch_weights / scale), -8, 7) * scale
    elif precision == 'int2':
        scale = ch_weights.abs().amax() / 1.5
        return torch.clamp(torch.round(ch_weights / scale), -1, 2) * scale
    elif precision == 'binary':
        scale = ch_weights.abs().mean()
        return torch.where(ch_weights >= 0, scale, -scale)
    return ch_weights

def apply_fabqvp_quantization(model, allocations):
    print("Applying FABQ-VP quantization...")
    quantized_state = {}
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and name in allocations:
            alloc = allocations[name]
            weights = module.weight.data.float()
            n_out, n_in = weights.shape
            
            quantized = torch.zeros_like(weights)
            for ch in range(n_out):
                prec = alloc.get(ch, 'int4')
                quantized[ch] = quantize_channel(weights[ch], prec)
            
            quantized_state[name] = quantized.half()
            module.weight.data = quantized.half()
    
    print("Quantization applied")
    return quantized_state

quantized_state = apply_fabqvp_quantization(model, allocations)

## 7. Evaluate Quantized Model Perplexity

In [ ]:
print("Computing quantized model perplexity...")
quantized_ppl = compute_perplexity(model, calib_ids)
print(f"FP16 Perplexity: {fp16_ppl:.4f}")
print(f"FABQ-VP Perplexity: {quantized_ppl:.4f}")
degradation = 100 * (quantized_ppl - fp16_ppl) / fp16_ppl
print(f"Degradation: {degradation:.2f}%")

## 8. Summary

**Results:** Compare FABQ-VP against baseline.

**Next Steps:**
1. Implement residual codebook for binary/int2 components
2. Add EBQ-style global allocation optimization